In [1]:
import gc
gc.collect()

14

In [2]:
import polars as pl

In [3]:
df_raw = pl.read_csv('final_user_course_features_AGENT.csv', null_values=['1,096', '1,390'])

In [4]:
# Шаг 1 — фильтруем только участников программы
df = df_raw.filter(
    pl.col("stats_m1_row_count").is_not_null() |
    pl.col("stats_m2_row_count").is_not_null() |
    pl.col("stats_m3_row_count").is_not_null()
)

# Шаг 2 — добавляем module ОБРАТНО В df (не в промежуточную переменную)
df = df.with_columns(
    pl.when(pl.col("stats_m1_row_count").is_not_null()).then(pl.lit("M1"))
      .when(pl.col("stats_m2_row_count").is_not_null()).then(pl.lit("M2"))
      .otherwise(pl.lit("M3"))
      .alias("module")
)

# Шаг 3 — теперь filter работает
m1_users = set(df.filter(pl.col("module") == "M1")["user_id"].to_list())
m2_users = set(df.filter(pl.col("module") == "M2")["user_id"].to_list())
overlap  = m1_users & m2_users
print(f"Пересечение user_id между M1 и M2: {len(overlap)}")

Пересечение user_id между M1 и M2: 1955


In [5]:
df.write_csv('M1_M4.csv')